This project implements four **closed-form stochastic models** to price **futures contracts on WTI crude oil**, with both in-sample estimation and out-of-sample evaluation. The models are applied on real data, and include error diagnostics and maturity-stratified performance analysis.

| Model | Description | SDE | Parameters |
|-------|-------------|-----|------------|
| 1. GBM (Gabillon) | Geometric Brownian Motion | dS = μ·Sₜ·dt + σ·Sₜ·dWₜ | δ |
| 2. OU (Schwartz) | Mean-reverting log process | dS = θ(μ − ln S)·dt + σ·S·dWₜ | θ, α, σ |
| 3. Modified Bessel | Square-root with nonlinear vol | dS = a·√S·dt + σ·S^(3/4)·dWₜ | a, σ |
| 4. Modified CIR | CIR drift with ¾ volatility | dS = (a·√S + b·S)·dt + σ·S^(3/4)·dWₜ | a, b, σ |

Each model provides an **analytical expression** for futures prices under no-arbitrage and deterministic interest rate assumptions.


🔹 In-Sample: `DATASETORG.xlsx`
- **Spot prices**: WTI (daily), 08/12/2022–19/11/2024
- **Futures prices**: 12 CME contracts (Jan–Dec 2024), 245 daily observations per contract
- **Interest rates**: Daily 10-year US Treasury rates from FRED
- **TTM**: Calculated in trading days

🔹 Out-of-Sample: `DATASETORG_OFS.xlsx`
- **Spot prices**: 10 trading days (23/10/2024 to 19/08/2024)
- **Futures prices**: 12 ICE contracts (expiring Jan–Dec 2025)
- **Interest rates**: 10-day sample of 10Y Treasury rates
- **TTM**: Provided as 10 × 12 matrix


In [ ]:
# preprocessing
import numpy as np
import pandas as pd

def load_data(filename):
    date = pd.read_excel(filename, sheet_name='date', header=None).values
    spot = pd.read_excel(filename, sheet_name='Spot', header=None).values
    futures = pd.read_excel(filename, sheet_name='Futures', header=None).values
    rates = pd.read_excel(filename, sheet_name='r', header=None).values
    ttm = pd.read_excel(filename, sheet_name='ttm', header=None).values
    return date, spot, futures, rates, ttm

def flatten_data(futures_table, spot_prices_table, interest_rates_table, time_to_maturity_table):
    f_flat = futures_table.flatten()
    s_flat = spot_prices_table.flatten()
    r_flat = interest_rates_table.flatten()
    ttm_flat = time_to_maturity_table.flatten()
    return f_flat, s_flat, r_flat, ttm_flat

def process_data(f, s, r, num_prices):
    num_contracts = f.shape[1]
    futures_table = np.full((num_prices, num_contracts), np.nan)
    interest_rates_table = np.full((num_prices, num_contracts), np.nan)
    spot_prices_table = np.full((num_prices, num_contracts), np.nan)
    time_to_maturity_table = np.full((num_prices, num_contracts), np.nan)

    for i in range(num_contracts):
        valid_indices = ~np.isnan(f[:, i])
        valid_futures = f[valid_indices, i]
        valid_rates = r[valid_indices, i]
        valid_spot = s[valid_indices, i]

        if len(valid_futures) >= num_prices:
            futures_table[:, i] = valid_futures[-num_prices:]
            interest_rates_table[:, i] = valid_rates[-num_prices:]
            spot_prices_table[:, i] = valid_spot[-num_prices:]
            time_to_maturity_table[:, i] = np.linspace(1, 0, num_prices)
        else:
            raise ValueError(f"Column {i+1} has fewer than {num_prices} valid rows. Check your data.")

    return futures_table, spot_prices_table, interest_rates_table, time_to_maturity_table

def load_data_oos(filename):
    spot = pd.read_excel(filename, sheet_name='Spot', header=None).values
    futures = pd.read_excel(filename, sheet_name='Futures', header=None).values
    rates = pd.read_excel(filename, sheet_name='r', header=None).values
    ttm = pd.read_excel(filename, sheet_name='ttm', header=None).values
    return spot, futures, rates, ttm



In [ ]:
# plotting
import matplotlib.pyplot as plt

def plot_model_fit(model_number, f_flat, s_flat, r_flat, ttm_flat, theta_total):
    
    #Generate plots to assess model fit:
    #1. Actual vs Predicted
    #2. Residuals vs TTM
    #3. Actual and Predicted over TTM#
    

    # Generate predicted futures prices
    if model_number == 1:
        f_pred = s_flat * np.exp((r_flat - theta_total[0]) * ttm_flat)

    elif model_number == 2:
        x_t = np.log(s_flat)
        kappa = theta_total[1]
        f_pred = np.exp(
            x_t * np.exp(-kappa * ttm_flat) +
            theta_total[0] * (1 - np.exp(-kappa * ttm_flat)) +
            (theta_total[2]**2 / (4 * kappa)) * (1 - np.exp(-2 * kappa * ttm_flat))
        )

    elif model_number == 3:
        a, sigma = theta_total
        f_pred = (
            s_flat + a * ttm_flat * np.sqrt(s_flat) +
            (a**2 * ttm_flat**2 / 4) * (1 - sigma**2 / (4 * a))
        )

    elif model_number == 4:
        a, b, sigma = theta_total
        f_pred = (
            s_flat * np.exp(b * ttm_flat) +
            (2 * a * np.sqrt(s_flat) / b) * (np.exp(b * ttm_flat) - np.exp(0.5 * b * ttm_flat)) +
            (a * (4 * a - sigma**2) / (4 * b**2)) * (np.exp(0.5 * b * ttm_flat) - 1)**2
        )
    else:
        raise ValueError("Invalid model number. Must be 1–4.")

    # Residuals
    residuals = f_flat - f_pred

    # Plot 1: Actual vs Predicted
    plt.figure(figsize=(6, 5))
    plt.scatter(f_flat, f_pred, color='blue', label='Predicted vs Actual')
    min_f, max_f = min(f_flat), max(f_flat)
    plt.plot([min_f, max_f], [min_f, max_f], 'r--', label='y = x')
    plt.xlabel("Actual Futures Prices")
    plt.ylabel("Predicted Futures Prices")
    plt.title(f"Model {model_number} – Actual vs Predicted Futures Prices")
    plt.grid(True)
    plt.legend()
    plt.show()

    # Plot 2: Residuals vs TTM
    plt.figure(figsize=(6, 5))
    plt.scatter(ttm_flat, residuals, color='blue')
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel("Time-to-Maturity (TTM)")
    plt.ylabel("Residuals (Actual - Predicted)")
    plt.title(f"Model {model_number} - Residuals vs Time-to-Maturity")
    plt.grid(True)
    plt.show()

    # Plot 3: Actual and Predicted over TTM
    plt.figure(figsize=(6, 5))
    plt.plot(ttm_flat, f_flat, 'b.', label='Actual Prices')
    plt.plot(ttm_flat, f_pred, 'r.', label='Predicted Prices')
    plt.xlabel("Time-to-Maturity (TTM)")
    plt.ylabel("Futures Prices")
    plt.title(f"Model {model_number} - Actual vs Predicted over Time-to-Maturity")
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_daily_rmse_comparison(daily_metrics_all_models):
    """
    Plots daily RMSE comparison across models.

    Parameters:
        daily_metrics_all_models (dict): Keys are model numbers, values are DataFrames with columns:
            - 'Day'
            - 'Daily_RMSE_Total'
    """
    plt.figure(figsize=(8, 5))
    for model_num, df in daily_metrics_all_models.items():
        plt.plot(df["Day"], df["Daily_RMSE_Total"], label=f"Model {model_num}", linewidth=1.5)

    plt.xlabel("Day")
    plt.ylabel("Daily RMSE")
    plt.title("Daily RMSE Comparison Across Models")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_class_rmse(maturity_classes, class_rmse, model_number):
    import matplotlib.pyplot as plt
    midpoints = 0.5 * (maturity_classes[:-1] + maturity_classes[1:])
    plt.figure(figsize=(6, 4))
    plt.bar(midpoints, class_rmse, width=np.diff(maturity_classes), edgecolor='black')
    plt.xlabel("Maturity Class (Years)")
    plt.ylabel("RMSE (Out-of-Sample)")
    plt.title(f"Model {model_number} – RMSE by Maturity Class (OOS)")
    plt.grid(True)
    plt.show()

def plot_class_rmse_grouped(maturity_classes, model_rmse_dict):
    models = sorted(model_rmse_dict.keys())
    num_classes = len(maturity_classes) - 1
    bar_width = 0.2
    x = np.arange(num_classes)

    plt.figure(figsize=(8, 5))
    for i, model in enumerate(models):
        rmse = model_rmse_dict[model]
        offset = (i - len(models)/2) * bar_width + bar_width/2
        plt.bar(x + offset, rmse, width=bar_width, label=f"Model {model}")

    class_labels = [f"[{maturity_classes[i]:.2f}, {maturity_classes[i+1]:.2f}]" for i in range(num_classes)]
    plt.xticks(x, class_labels)
    plt.xlabel("Maturity Class (Years)")
    plt.ylabel("RMSE (Out-of-Sample)")
    plt.title("RMSE by Maturity Class for All Models (OOS)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# models
from scipy.optimize import least_squares

# Dummy model functions (replace with real ones later)
def model1(f, s, r, ttm):
    return lambda theta: f - s * np.exp((r - theta[0]) * ttm)

def model2(f, s, ttm):
    return lambda theta: f - s * np.exp(theta[0] + theta[1]*ttm + theta[2]*ttm**2)

def model3(f, s, ttm):
    return lambda theta: f - (s + theta[0]*ttm*np.sqrt(s) + (theta[0]**2*ttm**2/4)*(1 - theta[1]**2/(4*theta[0])))

def model4(f, s, ttm):
    return lambda theta: f - (s * np.exp(theta[1]*ttm) +
        (2*theta[0]*np.sqrt(s)/theta[1])*(np.exp(theta[1]*ttm) - np.exp(0.5*theta[1]*ttm)) +
        (theta[0]*(4*theta[0] - theta[2]**2)/(4*theta[1]**2))*(np.exp(0.5*theta[1]*ttm) - 1)**2)

def run_model(model_number, f, s, r, ttm, theta0, lb, ub):
    if model_number == 1:
        objective = model1(f, s, r, ttm)
    elif model_number == 2:
        objective = model2(f, s, ttm)
    elif model_number == 3:
        objective = model3(f, s, ttm)
    elif model_number == 4:
        objective = model4(f, s, ttm)
    else:
        raise ValueError("Invalid model number.")

    result = least_squares(objective, x0=theta0, bounds=(lb, ub))
    return {
        "theta": result.x,
        "resnorm": result.cost * 2,
        "rmse": (2 * result.cost / len(f))**0.5
    }

def predict_prices(model_number, s, r, ttm, theta):
    if model_number == 1:
        return s * np.exp((r - theta[0]) * ttm)

    elif model_number == 2:
        x_t = np.log(s)
        kappa = theta[1]
        return np.exp(
            x_t * np.exp(-kappa * ttm) +
            theta[0] * (1 - np.exp(-kappa * ttm)) +
            (theta[2] ** 2 / (4 * kappa)) * (1 - np.exp(-2 * kappa * ttm))
        )

    elif model_number == 3:
        a, sigma = theta
        return (
            s + a * ttm * np.sqrt(s) +
            (a ** 2 * ttm ** 2 / 4) * (1 - sigma ** 2 / (4 * a))
        )

    elif model_number == 4:
        a, b, sigma = theta
        return (
            s * np.exp(b * ttm) +
            (2 * a * np.sqrt(s) / b) * (np.exp(b * ttm) - np.exp(0.5 * b * ttm)) +
            (a * (4 * a - sigma ** 2) / (4 * b ** 2)) * (np.exp(0.5 * b * ttm) - 1) ** 2
        )

    else:
        raise ValueError("Invalid model number")

In [ ]:
# metrics
def evaluate_maturity_classes(ttm, residuals, model_number):
    import numpy as np
    from plotting import plot_class_rmse

    maturity_classes = np.linspace(0, 1, 5)
    num_classes = len(maturity_classes) - 1
    class_sse = np.zeros(num_classes)
    class_rmse = np.zeros(num_classes)

    for j in range(num_classes):
        lower = maturity_classes[j]
        upper = maturity_classes[j + 1]
        ttm_mask = (ttm > lower) & (ttm <= upper)
        class_residuals = residuals[ttm_mask]

        if class_residuals.size > 0:
            class_sse[j] = np.sum(class_residuals**2)
            class_rmse[j] = np.sqrt(class_sse[j] / class_residuals.size)
        else:
            class_sse[j] = np.nan
            class_rmse[j] = np.nan

    print(f"📚 Maturity Class-Level Metrics (Model {model_number}):")
    for j in range(num_classes):
        print(f"  Class {j+1} [{maturity_classes[j]:.2f}, {maturity_classes[j+1]:.2f}]: "
              f"SSE = {class_sse[j]:.5f}, RMSE = {class_rmse[j]:.5f}")

    plot_class_rmse(maturity_classes, class_rmse, model_number)

from plotting import plot_class_rmse_grouped

def evaluate_maturity_classes_all_models(ttm, all_residuals_dict):
    maturity_classes = np.linspace(0, 1, 5)
    num_classes = len(maturity_classes) - 1
    model_rmse = {}

    for model_num, residuals in all_residuals_dict.items():
        class_rmse = np.zeros(num_classes)
        for j in range(num_classes):
            lower = maturity_classes[j]
            upper = maturity_classes[j + 1]
            ttm_mask = (ttm > lower) & (ttm <= upper)
            class_res = residuals[ttm_mask]

            if class_res.size > 0:
                class_rmse[j] = np.sqrt(np.sum(class_res**2) / class_res.size)
            else:
                class_rmse[j] = np.nan

        model_rmse[model_num] = class_rmse

    plot_class_rmse_grouped(maturity_classes, model_rmse)


In [ ]:
# main
from preprocessing import load_data, process_data, flatten_data, load_data_oos

# Load Excel data
path = 'DATASETORG.xlsx'
_, s, f, r, _ = load_data(path)

if r.shape[1] == 1:
    r = np.repeat(r, f.shape[1], axis=1)

if s.shape[1] == 1:
    s = np.repeat(s, f.shape[1], axis=1)

# Process data and flatten
f_table, s_table, r_table, ttm_table = process_data(f, s, r, 245)
f_flat, s_flat, r_flat, ttm_flat = flatten_data(f_table, s_table, r_table, ttm_table)

# Print shapes to confirm everything worked
print("Shapes:")
print("Futures:", f_flat.shape)
print("Spot:", s_flat.shape)
print("Rates:", r_flat.shape)
print("TTM:", ttm_flat.shape)

from models import run_model

# User choice
model_choice = 'all'  #choose between: 'all', 1, 2, 3, 4

# Parameter setup for each model
model_params = {
    1: {"theta0": [0.01], "lb": [-1], "ub": [1]},
    2: {"theta0": [0.1, 0.001, 0.01], "lb": [-1, 0, 0], "ub": [2, 1, 2]},
    3: {"theta0": [1.1, 0.01], "lb": [-2, 0], "ub": [20, 5]},
    4: {"theta0": [0.1, -0.1, 0.5], "lb": [0, -5, 0.5], "ub": [3.5, 0, 2.5]},
}

from plotting import plot_model_fit


# Run models -- you cant see all models plots at once, you have to choose between 1-4, else, you will just have the last model plot
results = {}  # ✅ always define this dictionary

# Run models
if model_choice == 'all':
    for i in range(1, 5):
        print(f"\n🔹 Running Model {i}")
        res = run_model(i, f_flat, s_flat, r_flat, ttm_flat, **model_params[i])
        results[i] = res
        print(f"Model {i} results:")
        for key, value in res.items():
            print(f"  {key}: {value}")
else:
    res = run_model(model_choice, f_flat, s_flat, r_flat, ttm_flat, **model_params[model_choice])
    results[model_choice] = res  # ✅ store single model result in the dictionary
    print(f"Model {model_choice} results:", res)
    
    # Plot the selected model
    theta_total = res["theta"]
    plot_model_fit(int(model_choice), f_flat, s_flat, r_flat, ttm_flat, theta_total)


from preprocessing import load_data_oos

s_oos, f_oos, r_oos, ttm_oos = load_data_oos("DATASETORG_OFS.xlsx")
# ✅ Flatten out-of-sample data
s_flat_oos = s_oos.flatten()
f_flat_oos = f_oos.flatten()
r_flat_oos = r_oos.flatten()
ttm_flat_oos = ttm_oos.flatten()

from models import predict_prices


print("\n📉 Out-of-Sample Evaluation:")

if model_choice == 'all':
    models_to_evaluate = range(1, 5)
else:
    models_to_evaluate = [model_choice]

for i in models_to_evaluate:
    theta = results[i]["theta"]
    f_pred_oos = predict_prices(i, s_flat_oos, r_flat_oos, ttm_flat_oos, theta)
    residuals_oos = f_flat_oos - f_pred_oos
    resnorm = np.sum(residuals_oos**2)
    rmse = np.sqrt(resnorm / len(f_flat_oos))

    print(f"Model {i}:")
    print(f"  Residual Norm (OOS): {resnorm:.4f}")
    print(f"  RMSE (OOS):          {rmse:.4f}")

# Store daily metrics per model
daily_metrics_all_models = {}

for i in models_to_evaluate:
    theta = results[i]["theta"]
    f_pred_oos = predict_prices(i, s_flat_oos, r_flat_oos, ttm_flat_oos, theta)

    # Reshape predicted and actual
    f_pred_matrix = f_pred_oos.reshape(f_oos.shape)
    f_actual_matrix = f_oos.reshape(f_oos.shape)

    # Residuals
    residuals_matrix = f_actual_matrix - f_pred_matrix

    # Initialize daily SSE/RMSE
    num_days, num_contracts = f_oos.shape
    daily_sse = np.zeros(num_days)
    daily_rmse = np.zeros(num_days)

    for day in range(num_days):
        residuals_day = residuals_matrix[day, :]
        residuals_day = residuals_day[~np.isnan(residuals_day)]
        daily_sse[day] = np.sum(residuals_day**2)
        if residuals_day.size > 0:
            daily_rmse[day] = np.sqrt(daily_sse[day] / residuals_day.size)
        else:
            daily_rmse[day] = np.nan

    # Store as DataFrame
    df = pd.DataFrame({
        "Day": np.arange(1, num_days + 1),
        "Daily_SSE_Total": daily_sse,
        "Daily_RMSE_Total": daily_rmse
    })
    daily_metrics_all_models[i] = df

# 🔽 Display all daily metrics
for i, df in daily_metrics_all_models.items():
    print(f"\n📊 Daily Metrics (Total Parameters) - Model {i}:")
    print(df)

from plotting import plot_daily_rmse_comparison

# ✅ Plot Daily RMSE Comparison
plot_daily_rmse_comparison(daily_metrics_all_models)    

from metrics import evaluate_maturity_classes
if model_choice in [1, 2, 3, 4]:
    evaluate_maturity_classes(ttm_flat_oos, residuals_oos, i)

from metrics import evaluate_maturity_classes_all_models

if model_choice == 'all':
    all_residuals_dict = {}
    for i in range(1, 5):
        theta = results[i]["theta"]
        f_pred_oos = predict_prices(i, s_flat_oos, r_flat_oos, ttm_flat_oos, theta)
        residuals_oos = f_flat_oos - f_pred_oos
        resnorm = np.sum(residuals_oos**2)
        rmse = np.sqrt(resnorm / len(f_flat_oos))

        print(f"Model {i}:")
        print(f"  Residual Norm (OOS): {resnorm:.4f}")
        print(f"  RMSE (OOS):          {rmse:.4f}")
        
        all_residuals_dict[i] = residuals_oos

    # 🔽 Single grouped bar chart
    evaluate_maturity_classes_all_models(ttm_flat_oos, all_residuals_dict)